# EXP-12: Hybrid Stacking — Transformers + TF-IDF + Threshold Tuning

**Competition:** spr-2026-mammography-report-classification

**Architecture:**
- **Level 0A:** BERTimbau Base (CLS pooling, FGM) → 7 OOF probs
- **Level 0B:** mDeBERTa-v3 Base (CLS pooling, FGM) → 7 OOF probs
- **Level 0C:** Calibrated LinearSVC on TF-IDF word+char → 7 OOF probs
- **Level 0D:** LightGBM on TF-IDF + dense clinical features → 7 OOF probs
- **Level 0E:** LogisticRegression L2 on TF-IDF → 7 OOF probs
- **Level 1:** LightGBM meta-learner on 35 OOF features
- **Post-processing:** CV-optimized threshold search + clinical guardrails

All transformer models use: FGM adversarial training, weighted Focal Loss, cosine schedule, fp16.

**Setup:** Add these datasets to notebook inputs:
- `cortese/bert-base-portuguese-cased`
- `cortese/mdeberta-v3-base`
- Competition data (auto-attached)

**Time budget:** ~90 min (2 transformers x 4 folds x 2 epochs + TF-IDF instant), well under 540 min GPU limit.

In [ ]:
# ============================================================
# Cell 1: Imports and device setup
# ============================================================
import os, gc, re, glob, time, hashlib, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion
from scipy.sparse import hstack, csr_matrix
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
except ImportError:
    !pip install -q lightgbm
    import lightgbm as lgb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')

In [ ]:
# ============================================================
# Cell 2: Config — paths, model configs, hyperparameters
# ============================================================
def find_input(pattern):
    """Locate a dataset directory under /kaggle/input by substring match.
    For model directories, verifies config.json exists (searches subdirs if needed)."""
    candidates = []
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if pattern in d.lower():
                candidates.append(os.path.join(root, d))
    # Return first candidate that has config.json (model dir) or any csv (comp dir)
    for c in candidates:
        if os.path.isfile(os.path.join(c, 'config.json')):
            return c
        if any(f.endswith('.csv') for f in os.listdir(c)):
            return c
    # If no config.json found at top level, search one level deeper
    for c in candidates:
        for sub in os.listdir(c):
            subpath = os.path.join(c, sub)
            if os.path.isdir(subpath) and os.path.isfile(os.path.join(subpath, 'config.json')):
                return subpath
    # Fallback: return first candidate
    return candidates[0] if candidates else None

# Competition data — search dynamically (path varies per Kaggle setup)
COMP_DIR = find_input('spr-2026-mammography')
assert COMP_DIR, (
    f'Competition data not found! Add the competition dataset to notebook inputs.\n'
    f'Contents of /kaggle/input: {os.listdir("/kaggle/input")}'
)
TRAIN_PATH = os.path.join(COMP_DIR, 'train.csv')
TEST_PATH  = os.path.join(COMP_DIR, 'test.csv')
print(f'Competition dir: {COMP_DIR}')

NUM_LABELS  = 7
N_SPLITS    = 4
SEED        = 42
WARMUP_RATIO = 0.1
WD           = 0.01
FOCAL_GAMMA  = 2.0

# Transformer model configs — only base models, 2 epochs for time budget
TRANSFORMER_MODELS = {
    'bertimbau_base': dict(
        path_pattern='bert-base-portuguese-cased',
        max_len=256,
        batch_size=32,
        grad_accum=1,
        epochs=2,
        lr=2e-5,
        pooling='cls',
        fgm_eps=0.5,
        enabled=True,
    ),
    'mdeberta': dict(
        path_pattern='mdeberta-v3-base',
        max_len=256,
        batch_size=32,
        grad_accum=1,
        epochs=2,
        lr=2e-5,
        pooling='cls',
        fgm_eps=0.5,
        enabled=True,
    ),
}

# Resolve transformer paths
for name, cfg in TRANSFORMER_MODELS.items():
    if cfg['enabled']:
        cfg['path'] = find_input(cfg['path_pattern'])
        if cfg['path']:
            # Verify model files exist
            has_model = (
                os.path.isfile(os.path.join(cfg['path'], 'config.json')) and
                (os.path.isfile(os.path.join(cfg['path'], 'model.safetensors')) or
                 os.path.isfile(os.path.join(cfg['path'], 'pytorch_model.bin')))
            )
            print(f'  {name}: {cfg["path"]} ({"OK" if has_model else "WARNING: no weights found!"})')
            if not has_model:
                # List contents to help debug
                print(f'    Contents: {os.listdir(cfg["path"])[:15]}')
        else:
            print(f'  {name}: NOT FOUND — disabling')
            cfg['enabled'] = False

enabled_transformers = {k: v for k, v in TRANSFORMER_MODELS.items() if v['enabled']}
print(f'\nEnabled transformers: {list(enabled_transformers.keys())}')

In [ ]:
# ============================================================
# Cell 3: Data loading and text preprocessing
# ============================================================
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f'Train: {train_df.shape} | Test: {test_df.shape}')
print(f'Class distribution:')
print(train_df['target'].value_counts().sort_index())

# --- Text cleaning for TF-IDF (aggressive normalization) ---
def clean_and_isolate_text(s):
    """Clean text for TF-IDF: lowercase, normalize whitespace, extract achados,
    replace measurements with tokens."""
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados:(.*?)(análise comparativa:|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'\b(cm|mm)\b', r' \1 ', s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def extract_dense_features(df):
    """Extract hand-crafted clinical features from report text."""
    features = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    features['report_length'] = text_col.apply(len)
    features['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    features['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    features['has_distortion'] = text_col.str.contains(r'distorção arquitetural', regex=True).astype(int)
    features['has_biopsy'] = text_col.str.contains(r'biopsy|biópsia|resultado de cine|carcinoma', regex=True).astype(int)
    features['has_calcification'] = text_col.str.contains(r'calcifica', regex=True).astype(int)
    features['has_nodule'] = text_col.str.contains(r'nódulo', regex=True).astype(int)
    features['has_asymmetry'] = text_col.str.contains(r'assimetria', regex=True).astype(int)
    features['has_birads_mention'] = text_col.str.contains(r'bi-?rads?', regex=True).astype(int)
    return csr_matrix(features.values)

# Light text cleaning for transformers (preserve more structure)
def clean_text(text):
    if pd.isna(text): return ''
    text = str(text).lower().strip()
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)
    return text

# Apply cleaning
train_df['clean'] = train_df['report'].apply(clean_text)
test_df['clean']  = test_df['report'].apply(clean_text)
train_df['tfidf_text'] = train_df['report'].apply(clean_and_isolate_text)
test_df['tfidf_text']  = test_df['report'].apply(clean_and_isolate_text)

# GroupKFold groups: MD5 hash of report text to avoid data leakage
groups = train_df['clean'].apply(lambda s: hashlib.md5(s.encode()).hexdigest()).values
labels = train_df['target'].values

gkf    = GroupKFold(n_splits=N_SPLITS)
splits = list(gkf.split(train_df, labels, groups))

print(f'\nUnique groups: {len(set(groups))}')
for fold, (tr_idx, va_idx) in enumerate(splits):
    print(f'  Fold {fold}: train={len(tr_idx)}, val={len(va_idx)}')

In [ ]:
# ============================================================
# Cell 4: Core components — ReportDataset, FlexClassifier,
#          FGM, WeightedFocalLoss, AttentionPooling
# ============================================================

class ReportDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class AttentionPooling(nn.Module):
    """Learned attention-weighted pooling over token hidden states."""
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)
    def forward(self, hidden_states, attention_mask):
        scores = self.attn(hidden_states).squeeze(-1)
        scores = scores.masked_fill(attention_mask == 0, -1e4)  # -1e4 safe for fp16
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)
        return (hidden_states * weights).sum(dim=1)


class FlexClassifier(nn.Module):
    """Transformer classifier with flexible pooling (cls, mean, attn)."""
    def __init__(self, model_path, num_labels, pooling='cls', drop_p=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_path)
        hidden = self.bert.config.hidden_size
        self.pooling_type = pooling
        if pooling == 'attn':
            self.pool = AttentionPooling(hidden)
        self.drop = nn.Dropout(drop_p)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kw):
        kw_bert = dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None:
            # mDeBERTa does not use token_type_ids — handle gracefully
            try:
                kw_bert['token_type_ids'] = token_type_ids
                out = self.bert(**kw_bert)
            except TypeError:
                del kw_bert['token_type_ids']
                out = self.bert(**kw_bert)
        else:
            out = self.bert(**kw_bert)

        hs = out.last_hidden_state
        if self.pooling_type == 'cls':
            pooled = hs[:, 0]
        elif self.pooling_type == 'mean':
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hs * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        elif self.pooling_type == 'attn':
            pooled = self.pool(hs, attention_mask)
        else:
            pooled = hs[:, 0]
        return self.classifier(self.drop(pooled))


class FGM:
    """Fast Gradient Method for adversarial training on embeddings."""
    def __init__(self, model, eps=0.5, emb_name='word_embeddings'):
        self.model = model
        self.eps = eps
        self.emb_name = emb_name
        self.backup = {}
    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0:
                    param.data.add_(self.eps * param.grad / norm)
    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


class WeightedFocalLoss(nn.Module):
    """Focal loss with per-class alpha weights for imbalanced classification."""
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer('alpha', alpha)
        self.gamma = gamma
    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, reduction='none')
        pt = torch.exp(-ce)
        return (self.alpha[labels] * ((1 - pt) ** self.gamma) * ce).mean()


def compute_class_weights(y, nc=7):
    """Inverse frequency weighting, normalized to mean=1."""
    c = np.bincount(y, minlength=nc).astype(float)
    w = len(y) / (nc * c + 1e-6)
    return torch.tensor(w / w.mean(), dtype=torch.float32)


def vram_mb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1e6
    return 0

print('Core components defined.')

In [ ]:
# ============================================================
# Cell 5: Transformer training function
# ============================================================

def train_transformer_cv(model_name, cfg):
    """Train a transformer model with GroupKFold CV.
    Returns OOF probabilities (n_train, 7) and test probabilities (n_test, 7)."""
    print(f'\n{"#"*60}')
    print(f'# Training: {model_name} (pooling={cfg["pooling"]}, '
          f'bs={cfg["batch_size"]}x{cfg["grad_accum"]}, ep={cfg["epochs"]}, lr={cfg["lr"]})')
    print(f'{"#"*60}')
    t0 = time.time()

    tokenizer = AutoTokenizer.from_pretrained(cfg['path'])
    oof_probs  = np.zeros((len(train_df), NUM_LABELS))
    test_probs = np.zeros((len(test_df), NUM_LABELS))
    fold_scores = []

    # Tokenize test once (small, only 4 rows)
    test_enc = tokenizer(
        test_df['report'].tolist(), truncation=True,
        padding='max_length', max_length=cfg['max_len'], return_tensors='pt',
    )

    for fold, (tr_idx, va_idx) in enumerate(splits):
        print(f'\n--- {model_name} Fold {fold} (train={len(tr_idx)}, val={len(va_idx)}, VRAM={vram_mb():.0f}MB) ---')

        tr_enc = tokenizer(
            train_df.iloc[tr_idx]['report'].tolist(), truncation=True,
            padding='max_length', max_length=cfg['max_len'], return_tensors='pt',
        )
        va_enc = tokenizer(
            train_df.iloc[va_idx]['report'].tolist(), truncation=True,
            padding='max_length', max_length=cfg['max_len'], return_tensors='pt',
        )

        tr_ds = ReportDataset(tr_enc, labels[tr_idx])
        va_ds = ReportDataset(va_enc)
        te_ds = ReportDataset(test_enc)

        tr_loader = DataLoader(tr_ds, batch_size=cfg['batch_size'], shuffle=True,
                               num_workers=2, pin_memory=True)
        va_loader = DataLoader(va_ds, batch_size=64, shuffle=False,
                               num_workers=2, pin_memory=True)
        te_loader = DataLoader(te_ds, batch_size=64, shuffle=False,
                               num_workers=2, pin_memory=True)

        # Load model and force fp32 (mDeBERTa has fp16 embeddings that break GradScaler)
        model = FlexClassifier(cfg['path'], NUM_LABELS, cfg['pooling']).to(DEVICE)
        model = model.float()
        print(f'  Model loaded. VRAM: {vram_mb():.0f} MB')

        alpha = compute_class_weights(labels[tr_idx]).to(DEVICE)
        criterion = WeightedFocalLoss(alpha, gamma=FOCAL_GAMMA)

        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=WD)
        total_steps = len(tr_loader) * cfg['epochs'] // cfg['grad_accum']
        warmup_steps = int(total_steps * WARMUP_RATIO)
        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        fgm = FGM(model, eps=cfg['fgm_eps'])
        scaler = torch.cuda.amp.GradScaler()

        best_f1 = 0
        best_oof = None
        best_state = None

        for epoch in range(cfg['epochs']):
            ep_start = time.time()
            model.train()
            total_loss = 0
            optimizer.zero_grad()

            for step, batch in enumerate(tr_loader):
                input_ids = batch['input_ids'].to(DEVICE)
                attn_mask = batch['attention_mask'].to(DEVICE)
                ttype_ids = batch.get('token_type_ids')
                if ttype_ids is not None:
                    ttype_ids = ttype_ids.to(DEVICE)
                lbl = batch['labels'].to(DEVICE)

                # Forward pass
                with torch.cuda.amp.autocast():
                    logits = model(input_ids, attn_mask, ttype_ids)
                    loss = criterion(logits, lbl) / cfg['grad_accum']
                scaler.scale(loss).backward()

                # FGM adversarial perturbation
                fgm.attack()
                with torch.cuda.amp.autocast():
                    logits_adv = model(input_ids, attn_mask, ttype_ids)
                    loss_adv = criterion(logits_adv, lbl) / cfg['grad_accum']
                scaler.scale(loss_adv).backward()
                fgm.restore()

                total_loss += loss.item() * cfg['grad_accum']

                if (step + 1) % cfg['grad_accum'] == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

            # Validation
            model.eval()
            va_probs = []
            with torch.no_grad():
                for batch in va_loader:
                    input_ids = batch['input_ids'].to(DEVICE)
                    attn_mask = batch['attention_mask'].to(DEVICE)
                    ttype_ids = batch.get('token_type_ids')
                    if ttype_ids is not None:
                        ttype_ids = ttype_ids.to(DEVICE)
                    with torch.cuda.amp.autocast():
                        logits = model(input_ids, attn_mask, ttype_ids)
                    va_probs.append(F.softmax(logits, dim=-1).cpu().float().numpy())

            va_probs = np.vstack(va_probs)
            f1 = f1_score(labels[va_idx], va_probs.argmax(1), average='macro')
            avg_loss = total_loss / len(tr_loader)
            ep_time = (time.time() - ep_start) / 60
            print(f'  Ep {epoch} | loss={avg_loss:.4f} | val_f1={f1:.4f} | '
                  f'time={ep_time:.1f}min | VRAM={vram_mb():.0f}MB')

            if f1 > best_f1:
                best_f1 = f1
                best_oof = va_probs.copy()
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        # Store OOF probabilities for this fold
        oof_probs[va_idx] = best_oof
        fold_scores.append(best_f1)

        # Test predictions using best checkpoint
        model.load_state_dict(best_state)
        model.eval()
        te_probs = []
        with torch.no_grad():
            for batch in te_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                attn_mask = batch['attention_mask'].to(DEVICE)
                ttype_ids = batch.get('token_type_ids')
                if ttype_ids is not None:
                    ttype_ids = ttype_ids.to(DEVICE)
                with torch.cuda.amp.autocast():
                    logits = model(input_ids, attn_mask, ttype_ids)
                te_probs.append(F.softmax(logits, dim=-1).cpu().float().numpy())
        test_probs += np.vstack(te_probs) / N_SPLITS

        # Cleanup to free VRAM
        del model, optimizer, scheduler, scaler, fgm, criterion, best_state
        del tr_enc, va_enc, tr_ds, va_ds, tr_loader, va_loader
        gc.collect()
        torch.cuda.empty_cache()

    oof_f1 = f1_score(labels, oof_probs.argmax(1), average='macro')
    elapsed = (time.time() - t0) / 60
    print(f'\n{model_name} DONE — OOF F1: {oof_f1:.4f} | '
          f'Mean fold: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f} | '
          f'Time: {elapsed:.1f} min')

    return oof_probs, test_probs, oof_f1

print('Training function defined.')

In [ ]:
# ============================================================
# Cell 6: Train both transformers
# ============================================================
all_oof  = {}   # model_name -> (n_train, 7)
all_test = {}   # model_name -> (n_test, 7)
all_f1   = {}   # model_name -> float

for name, cfg in enabled_transformers.items():
    oof, test, f1 = train_transformer_cv(name, cfg)
    all_oof[name]  = oof
    all_test[name] = test
    all_f1[name]   = f1

print('\n' + '='*60)
print('Transformer Results:')
for name, f1 in all_f1.items():
    print(f'  {name:25s} OOF F1 = {f1:.4f}')
print('='*60)

In [ ]:
# ============================================================
# Cell 7: TF-IDF models (SVM, LGB, LogReg) with OOF via GroupKFold
# ============================================================
print('Building TF-IDF features...')
t0 = time.time()

# --- Build TF-IDF vectorizers ---
tfidf_word = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), max_features=80000,
    sublinear_tf=True, min_df=2, max_df=0.95,
)
tfidf_char = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 5), max_features=80000,
    sublinear_tf=True, min_df=2, max_df=0.95,
)

# Fit on all train+test text for consistent vocabulary
all_text = pd.concat([train_df['tfidf_text'], test_df['tfidf_text']]).tolist()
tfidf_word.fit(all_text)
tfidf_char.fit(all_text)

X_word_train = tfidf_word.transform(train_df['tfidf_text'])
X_word_test  = tfidf_word.transform(test_df['tfidf_text'])
X_char_train = tfidf_char.transform(train_df['tfidf_text'])
X_char_test  = tfidf_char.transform(test_df['tfidf_text'])

# Combined TF-IDF features (word + char)
X_tfidf_train = hstack([X_word_train, X_char_train])
X_tfidf_test  = hstack([X_word_test, X_char_test])

# Dense clinical features
X_dense_train = extract_dense_features(train_df)
X_dense_test  = extract_dense_features(test_df)

# TF-IDF + dense (for LightGBM)
X_full_train = hstack([X_tfidf_train, X_dense_train])
X_full_test  = hstack([X_tfidf_test, X_dense_test])

print(f'TF-IDF word: {X_word_train.shape[1]} features')
print(f'TF-IDF char: {X_char_train.shape[1]} features')
print(f'Dense clinical: {X_dense_train.shape[1]} features')
print(f'Total TF-IDF+dense: {X_full_train.shape[1]} features')

# ========================================
# Model C: Calibrated LinearSVC on TF-IDF
# ========================================
print('\n--- Training Calibrated LinearSVC ---')
svm_oof  = np.zeros((len(train_df), NUM_LABELS))
svm_test = np.zeros((len(test_df), NUM_LABELS))

for fold, (tr_idx, va_idx) in enumerate(splits):
    base_svc = LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED)
    cal_svc = CalibratedClassifierCV(base_svc, cv=3, method='sigmoid')
    cal_svc.fit(X_tfidf_train[tr_idx], labels[tr_idx])
    svm_oof[va_idx] = cal_svc.predict_proba(X_tfidf_train[va_idx])
    svm_test += cal_svc.predict_proba(X_tfidf_test) / N_SPLITS
    print(f'  Fold {fold}: val_f1={f1_score(labels[va_idx], svm_oof[va_idx].argmax(1), average="macro"):.4f}')

svm_f1 = f1_score(labels, svm_oof.argmax(1), average='macro')
all_oof['svm_tfidf']  = svm_oof
all_test['svm_tfidf'] = svm_test
all_f1['svm_tfidf']   = svm_f1
print(f'SVM OOF F1: {svm_f1:.4f}')

# ========================================
# Model D: LightGBM on TF-IDF + dense
# ========================================
print('\n--- Training LightGBM ---')
lgb_oof  = np.zeros((len(train_df), NUM_LABELS))
lgb_test = np.zeros((len(test_df), NUM_LABELS))

LGB_PARAMS = dict(
    objective='multiclass', num_class=NUM_LABELS, metric='multi_logloss',
    learning_rate=0.1, num_leaves=63, min_child_samples=10,
    feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=3,
    class_weight='balanced', verbose=-1, seed=SEED, n_jobs=-1,
)

for fold, (tr_idx, va_idx) in enumerate(splits):
    dtrain = lgb.Dataset(X_full_train[tr_idx], label=labels[tr_idx])
    dvalid = lgb.Dataset(X_full_train[va_idx], label=labels[va_idx], reference=dtrain)
    bst = lgb.train(
        LGB_PARAMS, dtrain, num_boost_round=500,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)],
    )
    lgb_oof[va_idx] = bst.predict(X_full_train[va_idx])
    lgb_test += bst.predict(X_full_test) / N_SPLITS
    print(f'  Fold {fold}: val_f1={f1_score(labels[va_idx], lgb_oof[va_idx].argmax(1), average="macro"):.4f} '
          f'(best_iter={bst.best_iteration})')

lgb_f1 = f1_score(labels, lgb_oof.argmax(1), average='macro')
all_oof['lgb_tfidf']  = lgb_oof
all_test['lgb_tfidf'] = lgb_test
all_f1['lgb_tfidf']   = lgb_f1
print(f'LGB OOF F1: {lgb_f1:.4f}')

# ========================================
# Model E: Logistic Regression L2 on TF-IDF
# ========================================
print('\n--- Training LogisticRegression ---')
lr_oof  = np.zeros((len(train_df), NUM_LABELS))
lr_test = np.zeros((len(test_df), NUM_LABELS))

for fold, (tr_idx, va_idx) in enumerate(splits):
    lr_model = LogisticRegression(
        C=1.0, penalty='l2', solver='lbfgs', max_iter=1000,
        class_weight='balanced', multi_class='multinomial', random_state=SEED, n_jobs=-1,
    )
    lr_model.fit(X_tfidf_train[tr_idx], labels[tr_idx])
    lr_oof[va_idx] = lr_model.predict_proba(X_tfidf_train[va_idx])
    lr_test += lr_model.predict_proba(X_tfidf_test) / N_SPLITS
    print(f'  Fold {fold}: val_f1={f1_score(labels[va_idx], lr_oof[va_idx].argmax(1), average="macro"):.4f}')

lr_f1 = f1_score(labels, lr_oof.argmax(1), average='macro')
all_oof['logreg_tfidf']  = lr_oof
all_test['logreg_tfidf'] = lr_test
all_f1['logreg_tfidf']   = lr_f1
print(f'LogReg OOF F1: {lr_f1:.4f}')

elapsed = (time.time() - t0) / 60
print(f'\nAll TF-IDF models trained in {elapsed:.1f} min')

print('\n' + '='*60)
print('All Level-0 Models:')
for name, f1 in all_f1.items():
    print(f'  {name:25s} OOF F1 = {f1:.4f}')
print('='*60)

In [ ]:
# ============================================================
# Cell 8: Build stacking features, train LightGBM meta-learner
# ============================================================
model_names = list(all_oof.keys())
n_models    = len(model_names)

if n_models == 0:
    raise RuntimeError(
        'No models were trained. Check that datasets are attached to the notebook:\n'
        '  - cortese/bert-base-portuguese-cased\n'
        '  - cortese/mdeberta-v3-base\n'
        '  - Competition data'
    )

# Concatenate all OOF probabilities: (n_train, 7 * n_models)
X_stack_train = np.hstack([all_oof[m] for m in model_names])
X_stack_test  = np.hstack([all_test[m] for m in model_names])

print(f'Stacking features: {X_stack_train.shape[1]} ({n_models} models x {NUM_LABELS} classes)')
print(f'Models in stack: {model_names}')

# --- LightGBM Meta-Learner ---
META_PARAMS = dict(
    objective='multiclass', num_class=NUM_LABELS, metric='multi_logloss',
    learning_rate=0.05, num_leaves=15, min_child_samples=20,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=3,
    class_weight='balanced', verbose=-1, seed=SEED,
)

meta_oof  = np.zeros((len(train_df), NUM_LABELS))
meta_test = np.zeros((len(test_df), NUM_LABELS))
meta_fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    dtrain = lgb.Dataset(X_stack_train[tr_idx], label=labels[tr_idx])
    dvalid = lgb.Dataset(X_stack_train[va_idx], label=labels[va_idx], reference=dtrain)

    bst = lgb.train(
        META_PARAMS, dtrain, num_boost_round=500,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(100)],
    )
    meta_oof[va_idx] = bst.predict(X_stack_train[va_idx])
    meta_test += bst.predict(X_stack_test) / N_SPLITS

    fold_f1 = f1_score(labels[va_idx], meta_oof[va_idx].argmax(1), average='macro')
    meta_fold_scores.append(fold_f1)
    print(f'  Fold {fold}: meta val_f1={fold_f1:.4f} (best_iter={bst.best_iteration})')

meta_f1 = f1_score(labels, meta_oof.argmax(1), average='macro')
print(f'\nLGB Meta-Learner OOF F1: {meta_f1:.4f} '
      f'(folds: {np.mean(meta_fold_scores):.4f} +/- {np.std(meta_fold_scores):.4f})')

# Also compute simple weighted average for comparison
simple_avg = sum(all_oof[m] for m in model_names) / n_models
avg_f1 = f1_score(labels, simple_avg.argmax(1), average='macro')
print(f'Simple Average OOF F1: {avg_f1:.4f}')

# Choose best meta approach
if meta_f1 >= avg_f1:
    print('\n>>> Using LGB Meta-Learner for final predictions')
    final_oof_probs  = meta_oof
    final_test_probs = meta_test
else:
    print('\n>>> Using Simple Average for final predictions')
    final_oof_probs  = simple_avg
    final_test_probs = sum(all_test[m] for m in model_names) / n_models

baseline_preds = final_oof_probs.argmax(1)
baseline_f1 = f1_score(labels, baseline_preds, average='macro')
print(f'Baseline (argmax) OOF F1: {baseline_f1:.4f}')

In [ ]:
# ============================================================
# Cell 9: CV-optimized threshold search for rare classes
# ============================================================

def apply_thresholds(probs, thresholds):
    """Apply class-specific probability thresholds.
    For each class with a custom threshold, if its probability exceeds
    the threshold, override the argmax prediction.
    Process classes from highest to lowest (6 -> 5 -> 4) so that
    more severe classes take priority."""
    preds = probs.argmax(axis=1).copy()
    # Apply thresholds in descending class order (most severe first)
    for cls in sorted(thresholds.keys(), reverse=True):
        t = thresholds[cls]
        mask = probs[:, cls] >= t
        preds[mask] = cls
    return preds


def optimize_thresholds(oof_probs, y_true, classes_to_tune=[4, 5, 6]):
    """Greedy sequential threshold optimization for rare classes.
    Searches for the best probability threshold for each rare class
    that improves macro F1. Processes classes from most rare to least."""
    best_thresholds = {}
    best_f1 = f1_score(y_true, oof_probs.argmax(1), average='macro')
    print(f'Starting F1 (argmax): {best_f1:.4f}')

    for cls in sorted(classes_to_tune, reverse=True):  # 6 first, then 5, then 4
        best_t = None
        cls_best_f1 = best_f1
        for t in np.arange(0.05, 0.50, 0.01):
            preds = apply_thresholds(oof_probs, {**best_thresholds, cls: t})
            f1 = f1_score(y_true, preds, average='macro')
            if f1 > cls_best_f1:
                cls_best_f1 = f1
                best_t = t
        if best_t is not None:
            best_thresholds[cls] = best_t
            best_f1 = cls_best_f1
            print(f'  Class {cls}: threshold={best_t:.2f} -> F1={best_f1:.4f}')
        else:
            print(f'  Class {cls}: no threshold improvement found')

    return best_thresholds, best_f1


# Run threshold optimization
print('Optimizing thresholds for rare classes (4, 5, 6)...')
best_thresholds, threshold_f1 = optimize_thresholds(final_oof_probs, labels, classes_to_tune=[4, 5, 6])

print(f'\nThreshold optimization results:')
print(f'  Before: {baseline_f1:.4f}')
print(f'  After:  {threshold_f1:.4f}')
print(f'  Improvement: {threshold_f1 - baseline_f1:+.4f}')
print(f'  Best thresholds: {best_thresholds}')

# Also try tuning class 0
print('\nAlso trying class 0...')
extended_thresholds, extended_f1 = optimize_thresholds(
    final_oof_probs, labels, classes_to_tune=[0, 4, 5, 6]
)
print(f'  Extended thresholds: {extended_thresholds} -> F1={extended_f1:.4f}')

# Use whichever threshold set is better
if extended_f1 > threshold_f1:
    print('\n>>> Using extended thresholds (includes class 0)')
    final_thresholds = extended_thresholds
    final_threshold_f1 = extended_f1
else:
    print('\n>>> Using standard thresholds (classes 4,5,6 only)')
    final_thresholds = best_thresholds
    final_threshold_f1 = threshold_f1

print(f'Final thresholds: {final_thresholds}')
print(f'Final OOF F1 after thresholds: {final_threshold_f1:.4f}')

In [ ]:
# ============================================================
# Cell 10: Clinical guardrails + final submission
# ============================================================

def apply_safe_rules(row):
    """Clinical guardrail rules based on domain knowledge.
    These override model predictions when strong textual evidence is present."""
    text = str(row['report']).lower()
    pred = int(row['target'])

    # BI-RADS 6: confirmed malignancy
    if re.search(r'resultado de cine grau 3|carcinoma', text):
        return 6

    # BI-RADS 0: incomplete assessment
    if re.search(r'bi-?rads?\s*:?\s*0|categoria\s*:?\s*0', text):
        return 0

    # BI-RADS 5: highly suggestive of malignancy
    if 'espiculad' in text and 'distorção' in text and pred < 4:
        return 5

    # BI-RADS 4: suspicious
    if re.search(r'nódulo\s+espiculad', text) and pred < 4:
        return 4

    return pred


# --- Apply thresholds to test predictions ---
test_preds = apply_thresholds(final_test_probs, final_thresholds)

# Build submission dataframe
sub = pd.DataFrame({'ID': test_df['ID'], 'target': test_preds})
sub['report'] = test_df['report']  # Temporarily add for guardrail application

# --- Apply clinical guardrails ---
sub['target'] = sub.apply(apply_safe_rules, axis=1)
sub = sub[['ID', 'target']]  # Drop report column

# --- Validate OOF with guardrails ---
oof_with_thresh = apply_thresholds(final_oof_probs, final_thresholds)
oof_guard_df = train_df[['report']].copy()
oof_guard_df['target'] = oof_with_thresh
oof_guard_preds = oof_guard_df.apply(apply_safe_rules, axis=1).values
guardrail_f1 = f1_score(labels, oof_guard_preds, average='macro')

print('='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
print(f'\nLevel-0 individual models:')
for name, f1 in all_f1.items():
    print(f'  {name:25s} OOF F1 = {f1:.4f}')
print(f'\nMeta-learner (argmax):        {baseline_f1:.4f}')
print(f'+ Threshold tuning:           {final_threshold_f1:.4f}')
print(f'+ Clinical guardrails:        {guardrail_f1:.4f}')
print(f'\nThresholds used: {final_thresholds}')

print(f'\nSubmission distribution:')
print(sub['target'].value_counts().sort_index())

# Save submission
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(f'\nSubmission saved: {sub.shape}')
sub.head(10)